In [ ]:
# Cell 0: Install dependencies pinned to stable versions
!pip install -q transformers==4.44.2 sentence-transformers==3.0.1 langchain-huggingface langchain-community faiss-cpu datasets pandas torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 23.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 2.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the sourc

In [ ]:
# Cell 1: Load, Clean, and Balance Data for RAG
from datasets import load_dataset, concatenate_datasets
import pandas as pd
import re
import string
from datasets import Dataset

print("Loading CultureBank and MNLI...")
ds_cb = concatenate_datasets([load_dataset("SALT-NLP/CultureBank", split='tiktok'),
                              load_dataset("SALT-NLP/CultureBank", split='reddit')])
ds_mnli = load_dataset("multi_nli", split="train")

# Map columns (Using the exact dataset spelling 'actor_behavior')
ds_cb = ds_cb.rename_column("actor_behavior", "text").select_columns(["text"])
ds_mnli = ds_mnli.rename_column("premise", "text").select_columns(["text"])

# Filter MNLI for length
mnli_df = ds_mnli.to_pandas()
mnli_df['word_count'] = mnli_df['text'].apply(lambda x: len(str(x).split()))
long_mnli_df = mnli_df[mnli_df['word_count'] > 8].drop_duplicates(subset=["text"])
ds_generic = Dataset.from_pandas(long_mnli_df, preserve_index=False).remove_columns(["word_count"])

# Downsample to get exactly 1000 of each for our Vector DB Proof-of-Concept
ds_cb_sample = ds_cb.shuffle(seed=42).select(range(1000))
ds_generic_sample = ds_generic.shuffle(seed=42).select(range(1000))

# Universal Sanitizer
def sanitize_and_debias(example):
    text = str(example["text"])
    words_to_remove = r'\b(customary|common practice|norm|culture|tradition|expected)\b'
    text = re.sub(words_to_remove, '', text, flags=re.IGNORECASE)
    location_pattern = r'^In\s+[A-Z][a-z]+\s*(culture|society)?\s*,'
    text = re.sub(location_pattern, '', text)
    example["text"] = text.rstrip(string.punctu ation).strip()
    example["text"] = re.sub(r'\s+', ' ', example["text"])
    return example

print("Sanitizing documents...")
ds_cb_sample = ds_cb_sample.map(sanitize_and_debias)
ds_generic_sample = ds_generic_sample.map(sanitize_and_debias)

print(f"Data ready! {len(ds_cb_sample)} Norms and {len(ds_generic_sample)} Generics.")

Loading CultureBank and MNLI...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/186 [00:00<?, ?B/s]

tiktok/culturebank_tiktok.csv:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

reddit/culturebank_reddit.csv:   0%|          | 0.00/20.7M [00:00<?, ?B/s]

...

Generating validation_mismatched split:   0%|          | 0/9832 [00:00<?, ? examples/s]

Sanitizing documents...


Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Data ready! 1000 Norms and 1000 Generics.


In [ ]:
# Cell 2: Build the FAISS Vector Database (Hardcoded BERT Implementation)
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from transformers import BertTokenizer, BertModel
import torch
from typing import List

print("1. Tagging and creating Document objects...")
documents = []

for item in ds_cb_sample:
    doc_text = f"[Source: Norm] {item['text']}"
    documents.append(Document(page_content=doc_text))

for item in ds_generic_sample:
    doc_text = f"[Source: Generic Fact] {item['text']}"
    documents.append(Document(page_content=doc_text))

print("2. Initializing Hardcoded PyTorch Embedding Engine...")

class RawPyTorchEmbeddings(Embeddings):
    def __init__(self, model_name: str):
        # Hardcoding BERT classes stops the library from searching for missing "Chat Templates"
        self.tokenizer = BertTokenizer.from_pretrained(model_name)
        self.model = BertModel.from_pretrained(model_name)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model.to(self.device)
        self.model.eval()

    def _mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        batch_size = 32
        all_embeddings = []

        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i+batch_size]
            encoded_input = self.tokenizer(batch_texts, padding=True, truncation=True, return_tensors='pt', max_length=128).to(self.device)

            with torch.no_grad():
                model_output = self.model(**encoded_input)

            batch_embeddings = self._mean_pooling(model_output, encoded_input['attention_mask'])
            all_embeddings.extend(batch_embeddings.cpu().tolist())

        return all_embeddings

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

embeddings = RawPyTorchEmbeddings("sentence-transformers/all-MiniLM-L6-v2")

print("3. Embedding 2,000 documents into FAISS (This will take a minute)...")
vector_db = FAISS.from_documents(documents, embeddings)

# Fetch the top 3 closest matches during search
retriever = vector_db.as_retriever(search_kwargs={"k": 3})
print("Vector Database successfully built!")

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

1. Tagging and creating Document objects...
2. Initializing Hardcoded PyTorch Embedding Engine...


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

3. Embedding 2,000 documents into FAISS (This will take a minute)...
Vector Database successfully built!


In [ ]:
# Cell 3: Orchestrate the LLM and RAG Pipeline
from transformers import pipeline
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

print("Loading Flan-T5 Reasoning Engine...")

hf_pipeline = pipeline(
    "text2text-generation",
    model="google/flan-t5-base",
    max_new_tokens=10,
    device_map="auto"
)
llm = HuggingFacePipeline(pipeline=hf_pipeline)

# System prompt forcing the LLM to rely on retrieved context
template = """You are a classification AI. Your task is to classify the INPUT SENTENCE as "Norm" or "Generic".

Look at the SIMILAR DOCUMENTS retrieved from our database.
- If the majority of similar documents are tagged [Source: Norm], classify the input as "Norm".
- If the majority are tagged [Source: Generic Fact], classify the input as "Generic".
- Read the semantic meaning of the documents to guide your final answer.

SIMILAR DOCUMENTS:
{context}

INPUT SENTENCE:
{question}

CLASSIFICATION (Reply ONLY with "Norm" or "Generic"):"""

custom_rag_prompt = PromptTemplate.from_template(template)

def format_docs(docs):
    return "\n".join(f"- {doc.page_content}" for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | custom_rag_prompt
    | llm
    | StrOutputParser()
)
print("RAG Pipeline Ready!")

Loading Flan-T5 Reasoning Engine...


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

RAG Pipeline Ready!


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
# Cell 4: Evaluate RAG on the Adversarial Sentences
import pandas as pd

test_sentences = [
    "Tipping 20 percent at restaurants is considered polite",
    "The mitochondria is the powerhouse of the cell",
    "Employees are expected to arrive by 9:00 AM",
    "The Eiffel Tower is located in Paris, France",
    "Water must reach 100 degrees Celsius to boil",
    "Guests bring gifts to weddings",
    "Men take off their hats when entering a church",
    "A triangle has exactly three sides and three angles",
    "You should brush your teeth twice a day",
    "Planets orbit the sun due to gravitational pull"
]
expected_labels = ["Norm", "Generic", "Norm", "Generic", "Generic", "Norm", "Norm", "Generic", "Norm", "Generic"]

print("Running RAG Evaluation...\n")

results = []
for i, text in enumerate(test_sentences):

    # Check the database context
    retrieved = retriever.invoke(text)
    context_str = " | ".join([doc.page_content[:45] + "..." for doc in retrieved])

    # Get Final LLM Prediction
    prediction = rag_chain.invoke(text).strip()

    results.append({
        "Sentence": text,
        "Expected": expected_labels[i],
        "RAG Prediction": prediction,
        "Top Retrieval Snippet": context_str
    })

# Format and style the output table
df_results = pd.DataFrame(results)

def highlight_rag_errors(row):
    colors = ['' for _ in row]
    col_idx = row.index.get_loc("RAG Prediction")
    if row["RAG Prediction"] != row["Expected"]:
        colors[col_idx] = 'background-color: lightcoral'
    else:
        colors[col_idx] = 'background-color: lightgreen'
    return colors

styled_df = df_results.style.apply(highlight_rag_errors, axis=1)
display(styled_df)

Running RAG Evaluation...



,Sentence,Expected,RAG Prediction,Top Retrieval Snippet
0,Tipping 20 percent at restaurants is considered polite,Norm,Generic,"[Source: Norm] tipping is not common, but can... | [Source: Norm] waitstaff provide attentive, r... | [Source: Norm] engage in tip-based earnings, ..."
1,The mitochondria is the powerhouse of the cell,Generic,Generic,[Source: Generic Fact] You've seen this sort ... | [Source: Generic Fact] you know that's fine a... | [Source: Generic Fact] Sometimes funny and so...
2,Employees are expected to arrive by 9:00 AM,Norm,Norm,[Source: Norm] value punctuality and have a r... | [Source: Norm] adhere to a flexible and delay... | [Source: Generic Fact] GAO also expects that ...
3,"The Eiffel Tower is located in Paris, France",Generic,Generic,[Source: Generic Fact] Climb the 15th-century... | [Source: Generic Fact] The dimensions of the ... | [Source: Generic Fact] You should also look a...
4,Water must reach 100 degrees Celsius to boil,Generic,Generic,"[Source: Norm] boil water with spices, add co... | [Source: Generic Fact] At the Stabian Baths (... | [Source: Generic Fact] The second choice is a..."
5,Guests bring gifts to weddings,Norm,Generic,"[Source: Norm] exchange gifts, including mugs... | [Source: Norm] offer gifts, including money, ... | [Source: Norm] offer food and drinks, includi..."
6,Men take off their hats when entering a church,Norm,Generic,[Source: Generic Fact] This may have been to ... | [Source: Generic Fact] You will also be caugh... | [Source: Generic Fact] Two good places to wit...
7,A triangle has exactly three sides and three angles,Generic,Generic,"[Source: Generic Fact] And we did it with the... | [Source: Generic Fact] The executive will hav... | [Source: Generic Fact] For example, if the pu..."
8,You should brush your teeth twice a day,Norm,Norm,"[Source: Generic Fact] Please throw away your... | [Source: Norm] practice daily showering, use ... | [Source: Generic Fact] of it being a little b..."
9,Planets orbit the sun due to gravitational pull,Generic,Generic,[Source: Generic Fact] While in the private s... | [Source: Generic Fact] there was another uh a... | [Source: Generic Fact] And we did it with the...


In [ ]:
# Cell 5: Formal Accuracy Evaluation on Unseen Data
from sklearn.metrics import accuracy_score, classification_report
from tqdm.auto import tqdm

print("1. Fetching unseen test data...")
# We grab rows 1000 to 1100. These were NOT put into the FAISS database!
test_norms = ds_cb.shuffle(seed=42).select(range(1000, 1100))
test_generics = ds_generic.shuffle(seed=42).select(range(1000, 1100))

# Apply our universal sanitizer so there is no punctuation bias
test_norms = test_norms.map(sanitize_and_debias)
test_generics = test_generics.map(sanitize_and_debias)

# Combine them into lists (FIXED: Cast the Hugging Face columns to Python lists)
test_sentences = list(test_norms['text']) + list(test_generics['text'])
true_labels = ["Norm"] * 100 + ["Generic"] * 100

print(f"2. Testing RAG on {len(test_sentences)} completely new sentences...")
print("(This may take a few minutes as the LLM reads and reasons through each one)\n")

predictions = []
for text in tqdm(test_sentences, desc="Evaluating"):
    # Ask the RAG pipeline to classify the sentence
    pred = rag_chain.invoke(text).strip()

    # We strictly enforce the output format just in case the LLM includes a period or extra word
    if "Norm" in pred:
        predictions.append("Norm")
    else:
        predictions.append("Generic")

print("\n" + "="*40)
print("--- RAG FINAL TEST RESULTS ---")
print("="*40)

# Calculate and print the exact metrics
accuracy = accuracy_score(true_labels, predictions)
print(f"Overall Accuracy: {accuracy:.2%}\n")

print(classification_report(true_labels, predictions, target_names=["Generic", "Norm"]))

1. Fetching unseen test data...


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

2. Testing RAG on 200 completely new sentences...
(This may take a few minutes as the LLM reads and reasons through each one)



Evaluating:   0%|          | 0/200 [00:00<?, ?it/s]

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset
Token indices sequence length is longer than the specified maximum sequence length for this model (558 > 512). Running this sequence through the model will result in indexing errors



--- RAG FINAL TEST RESULTS ---
Overall Accuracy: 59.50%

              precision    recall  f1-score   support

     Generic       0.55      0.97      0.71       100
        Norm       0.88      0.22      0.35       100

    accuracy                           0.59       200
   macro avg       0.72      0.59      0.53       200
weighted avg       0.72      0.59      0.53       200

